In [7]:
from ultralytics import YOLO
import os, json, random, shutil
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

In [10]:
def adjust_brightness_contrast(image, brightness_factor=None, contrast_factor=None):
    """
    Adjust brightness and contrast of an image
    """
    if brightness_factor is None:
        brightness_factor = random.uniform(0.6, 1.4)
    if contrast_factor is None:
        contrast_factor = random.uniform(0.7, 1.3)

    # Adjust brightness
    bright_img = cv2.multiply(image, brightness_factor)

    # Adjust contrast
    mean = np.mean(bright_img)
    contrast_img = cv2.addWeighted(bright_img, contrast_factor, mean, 0, 0)

    # Clip values to valid range [0, 255]
    final_img = np.clip(contrast_img, 0, 255).astype(np.uint8)

    return final_img, brightness_factor, contrast_factor


def rotate_point(x, y, angle, cx=0.5, cy=0.5):
    """Rotate a point around a center point"""
    angle_rad = np.radians(angle)
    x_shifted = x - cx
    y_shifted = y - cy
    x_rotated = x_shifted * np.cos(angle_rad) - y_shifted * np.sin(angle_rad)
    y_rotated = x_shifted * np.sin(angle_rad) + y_shifted * np.cos(angle_rad)
    x_final = x_rotated + cx
    y_final = y_rotated + cy
    return x_final, y_final


def rotate_yolo_annotation(annotation, angle):
    """Rotate YOLO format annotation"""
    class_id = annotation[0]
    x_center, y_center = rotate_point(annotation[1], annotation[2], angle)

    width = annotation[3]
    height = annotation[4]

    if angle in [90, 270]:
        width, height = height, width

    x_center = np.clip(x_center, 0, 1)
    y_center = np.clip(y_center, 0, 1)

    return [class_id, x_center, y_center, width, height]


def ajouter_brouillard(image, intensite=0.5):
    """add fog effect to an image"""
    image = image.astype(np.float32) / 255.0

    hauteur, largeur = image.shape[:2]
    bruit = np.random.normal(loc=0.5, scale=0.5, size=(hauteur, largeur)).astype(np.float32)
    brouillard = cv2.GaussianBlur(bruit, (0, 0), sigmaX=max(1.0, hauteur/10), sigmaY=max(1.0, largeur/10))

    brouillard = cv2.normalize(brouillard, None, 0, 1, cv2.NORM_MINMAX)
    brouillard = brouillard[:, :, np.newaxis]
    brouillard = np.repeat(brouillard, 3, axis=2)

    image_brouillard = cv2.addWeighted(image, 1 - float(intensite), brouillard, float(intensite), 0)
    image_brouillard = (np.clip(image_brouillard, 0.0, 1.0) * 255).astype(np.uint8)

    return image_brouillard

def build_tilt_transform_matrix(w, h, tilt_deg):
    """Return a 3x3 perspective transform matrix that simulates an X-axis tilt.
    tilt_deg: positive tilts the top edge away (makes top narrower), negative brings it closer.
    """
    # Limit tilt to avoid degenerate transforms
    max_tilt = 15  # reduced from 60 to 15 degrees
    tilt_deg = np.clip(tilt_deg, -max_tilt, max_tilt)

    # Strength parameter derived from angle; map degrees to a vertical shift factor
    t = np.tan(np.radians(tilt_deg)) * 0.5  # scale down to reasonable perspective

    # Define source points (corners of the image)
    src = np.array([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]], dtype=np.float32)

    # Calculate how much to move top edge inward (toward center) based on t
    dx = t * w
    # Move top-left and top-right towards center by dx
    dst_top_left = np.array([dx, 0])
    dst_top_right = np.array([w - 1 - dx, 0])

    # Optionally scale vertical positions to simulate foreshortening
    # y_scale reduces the height perceived for the top when tilting away
    y_scale = 1.0 - abs(t) * 0.5
    dst_bottom_right = np.array([w - 1, (h - 1) * y_scale + (1 - y_scale) * h * 0.5])
    dst_bottom_left = np.array([0, (h - 1) * y_scale + (1 - y_scale) * h * 0.5])

    dst = np.vstack([dst_top_left, dst_top_right, dst_bottom_right, dst_bottom_left]).astype(np.float32)

    # Compute perspective transform
    M = cv2.getPerspectiveTransform(src, dst)
    return M


def transform_point(pt, M):
    """Apply homography M to a single point (x, y)."""
    x, y = pt
    vec = np.array([x, y, 1.0], dtype=np.float32)
    tx, ty, tz = M.dot(vec)
    if tz == 0:
        return (0, 0)
    return (tx / tz, ty / tz)


def yolo_to_corners(xc, yc, w, h, img_w, img_h):
    """Convert YOLO normalized bbox to pixel corner coordinates (4 corners)."""
    x_center = xc * img_w
    y_center = yc * img_h
    bw = w * img_w
    bh = h * img_h
    x1 = x_center - bw / 2.0
    y1 = y_center - bh / 2.0
    x2 = x_center + bw / 2.0
    y2 = y_center + bh / 2.0
    return [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]


def corners_to_yolo(corners, img_w, img_h, class_id=0):
    """Convert transformed corners (pixel coords) to YOLO normalized bbox (class, xc, yc, w, h)."""
    xs = [c[0] for c in corners]
    ys = [c[1] for c in corners]
    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    # Clamp to image bounds
    x_min = max(0, min(x_min, img_w - 1))
    x_max = max(0, min(x_max, img_w - 1))
    y_min = max(0, min(y_min, img_h - 1))
    y_max = max(0, min(y_max, img_h - 1))

    bw = x_max - x_min
    bh = y_max - y_min
    if bw <= 0 or bh <= 0:
        return None  # invalid box after transform

    xc = (x_min + x_max) / 2.0 / img_w
    yc = (y_min + y_max) / 2.0 / img_h
    nw = bw / img_w
    nh = bh / img_h
    return [class_id, xc, yc, nw, nh]



def creer_dataset_augmente(
    dossier_images_source,
    dossier_labels_source,
    dossier_images_destination,
    dossier_labels_destination,
    n_rotations=200,
    n_fog=250,
    n_brightness=200,
    n_rotations_x=100,
    deg_x=15
):
      # Creation of destination folders if they do not exist
    Path(dossier_images_destination).mkdir(parents=True, exist_ok=True)
    Path(dossier_labels_destination).mkdir(parents=True, exist_ok=True)

    # convertion in Path objects
    source_images = Path(dossier_images_source)
    source_labels = Path(dossier_labels_source)
    dest_images = Path(dossier_images_destination)
    dest_labels = Path(dossier_labels_destination)

    # take all images from source folder
    extensions_valides = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    toutes_images = [f for f in source_images.iterdir()
                     if f.suffix.lower() in extensions_valides]

    print(f" Number of images found :  {len(toutes_images)}")

    compteur = {
        'originales': 0,
        'rotations': 0,
        'fog': 0,
        'brightness': 0
    }

    # 1. Copie of the original images
    print("\n Copy of the original images...")
    for img_path in toutes_images:
        # Copie
        shutil.copy2(img_path, dest_images / img_path.name)

        # Copie if label exists
        label_path = source_labels / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(label_path, dest_labels / f"{img_path.stem}.txt")

        compteur['originales'] += 1

    print(f" {compteur['originales']} copied original images")

    # 2. ADD ROTATIONS
    print(f"\n add {n_rotations} rotated images...")
    images_pour_rotation = random.sample(toutes_images, min(n_rotations, len(toutes_images)))
    angles = [90, 180, 270]

    for img_path in images_pour_rotation:
        angle = random.choice(angles)

        # lecture
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        # rotation
        height, width = img.shape[:2]
        center = (width // 2, height // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated_img = cv2.warpAffine(img, rotation_matrix, (width, height))

        # saving
        nouveau_nom = f"{img_path.stem}_rot{angle}{img_path.suffix}"
        cv2.imwrite(str(dest_images / nouveau_nom), rotated_img)

        # copie label if exists
        label_path = source_labels / f"{img_path.stem}.txt"
        if label_path.exists():
            with open(label_path, 'r') as f:
                annotations = f.readlines()

            rotated_annotations = []
            for ann in annotations:
                ann_parts = list(map(float, ann.strip().split()))
                rotated_ann = rotate_yolo_annotation(ann_parts, angle)
                rotated_annotations.append(' '.join(map(str, rotated_ann)))

            with open(dest_labels / f"{img_path.stem}_rot{angle}.txt", 'w') as f:
                f.write('\n'.join(rotated_annotations))

        compteur['rotations'] += 1

    print(f" {compteur['rotations']} images with rotation added")

    # 3. ADD IMAGES WITH BROUILLARD
    print(f"\n  add {n_fog} images with fog...")
    images_pour_fog = random.sample(toutes_images, min(n_fog, len(toutes_images)))

    for img_path in images_pour_fog:
        # Lire l'image
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        # add fog
        intensite = random.uniform(0.2, 0.7)
        img_fog = ajouter_brouillard(img, intensite=intensite)

        # Saving
        nouveau_nom = f"fog_{img_path.name}"
        cv2.imwrite(str(dest_images / nouveau_nom), img_fog)

        # copi label if exists
        label_path = source_labels / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(label_path, dest_labels / f"fog_{img_path.stem}.txt")

        compteur['fog'] += 1

    print(f" {compteur['fog']} images with fog added")

    # 4. ADD IMAGES WITH MODIFIED BRIGHTNESS/CONTRAST
    print(f"\n Add {n_brightness} images with modified brightness/contrast...")
    images_pour_brightness = random.sample(toutes_images, min(n_brightness, len(toutes_images)))

    for img_path in images_pour_brightness:
        # lecture
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        # modifier brightness/contrast
        adjusted_img, brightness, contrast = adjust_brightness_contrast(img)

        # Sauvegarder
        nouveau_nom = f"bright_{img_path.name}"
        cv2.imwrite(str(dest_images / nouveau_nom), adjusted_img)


        label_path = source_labels / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(label_path, dest_labels / f"bright_{img_path.stem}.txt")

        compteur['brightness'] += 1

    print(f" {compteur['brightness']} images with added brightness/contrast")

    print(f"\n Add {n_rotations_x} images with modified brightness/contrast...")
     #Read image
    img = cv2.imread(str(dossier_images_source))
    if img is None:
        print(f"Failed to read image: {dossier_images_source}")
        return False
    h, w = img.shape[:2]
    tilt_deg = random.uniform(-deg_x, deg_x)
    # Build transform matrix and apply
    M = build_tilt_transform_matrix(w, h, tilt_deg)
    tilted = cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_LINEAR)

    # Process annotation file if it exists
    transformed_annotations = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            annotations = [line.strip() for line in f if line.strip()]
        for ann in annotations:
            parts = ann.split()
            class_id = int(parts[0])
            vals = list(map(float, parts[1:5]))
            corners = yolo_to_corners(vals[0], vals[1], vals[2], vals[3], w, h)
            transformed_corners = [transform_point(c, M) for c in corners]
            new_box = corners_to_yolo(transformed_corners, w, h, class_id)
            if new_box is not None:
                transformed_annotations.append(new_box)
    else:
        # No label file; that's okay
        pass

    # Write output image
    cv2.imwrite(str(dossier_images_destination), tilted)

    # Write labels
    if transformed_annotations:
        with open(dossier_labels_destination, 'w') as f:
            for ann in transformed_annotations:
                f.write(' '.join(map(str, ann)) + '\n')

    # RÉSUMÉ FINAL
    total = sum(compteur.values())
    print("\n" + "="*60)
    print("Resume of dataset augmentation:")
    print("="*60)
    print(f"original images:        {compteur['originales']}")
    print(f"rotated images:     {compteur['rotations']}")
    print(f"fog images:   {compteur['fog']}")
    print(f"images with brigthness:   {compteur['brightness']}")
    print(f"{'─'*60}")
    print(f"TOTAL:                    {total} images")
    print("="*60)
    print(f"\nDataset creates in: {dossier_images_destination}")
    print(f" Labels creates in: {dossier_labels_destination}")


# UTILISATION
if __name__ == "__main__":
    creer_dataset_augmente(
        dossier_images_source="./dataset/images/train",
        dossier_labels_source="./dataset/labels/train",
        dossier_images_destination="./dataset_augmente/images/train",
        dossier_labels_destination="./dataset_augmente/labels/train",
        n_rotations=200,
        n_fog=250,
        n_brightness=200,
        n_rotations_x=100
    )
    create_dataset_augmente(
        dossier_images_source="./dataset/images/valid",
        dossier_labels_source="./dataset/labels/valid",
        dossier_images_destination="./dataset_augmente/images/valid",
        dossier_labels_destination="./dataset_augmente/labels/valid",
        n_rotations=50,
        n_fog=50,
        n_brightness=50,
        n_rotations_x=50
    )
    creer_dataset_augmente(
        dossier_images_source="./dataset/images/test",
        dossier_labels_source="./dataset/labels/test",
        dossier_images_destination="./dataset_augmente/images/test",
        dossier_labels_destination="./dataset_augmente/labels/test",
        n_rotations=50,
        n_fog=50,
        n_brightness=50,
        n_rotations_x=50
    )

 Number of images found :  1386

 Copy of the original images...
 1386 copied original images

 add 200 rotated images...
 200 images with rotation added

  add 250 images with fog...
 250 images with fog added

 Add 200 images with modified brightness/contrast...
 200 images with added brightness/contrast

 Add 100 images with modified brightness/contrast...


PermissionError: [Errno 13] Permission denied: './dataset/images/train'

In [ ]:
model = YOLO("yolo11n.pt")

# training the model
results = model.train(
    data="data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    name="sard2_yolo11_augmented",
    project="runs/train",
    workers=0,
)